# 00 — Diagnostic tools

**This notebook is NOT meant to be run end-to-end.** It is a toolbox of independent diagnostic cells.

## Workflow

1. Run the **Setup** cell once (imports, helpers).
2. Set `PARTICIPANT` and `CONDITION` at the top of the section you want to use.
3. Run only the cells of that section.

## Sections

- **A — Raw inspection**: load a `.mat`, check marker visibility, plot Dos Z trajectories.
- **B — Crop window check**: visualise where `crop_start_s` / `crop_end_s` fall on the raw signal.
- **C — Sanity check**: cadence, dominant frequency, amplitude on a processed trial.
- **D — Despike diagnostic**: number of spikes flagged, signal before/after despike.
- **E — Signal quality plot**: full preprocessing chain (raw → despiked → EMD → filtered → final).
- **F — 2D trajectory**: sacrum XY plotted on top of the sand bed rectangle.

Replaces the legacy notebooks: `00_explore_raw`, `02b_visual_check_crops`, `02c_sanity_check`, `02d_diagnose_anomalies`.

## Setup (run once)

In [ ]:
from resilience import paths, participants, config
from resilience.io import loader, writer
from resilience.processing import preprocess, detection
from resilience.viz import plots

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from pathlib import Path
%matplotlib inline

pd.set_option('display.float_format', '{:.3f}'.format)

# Helpers
def load_raw(participant, condition):
    """Load raw .mat and return trajectories dict."""
    raw_path = paths.raw_file(participant, condition)
    data, fmt = loader.load_mat(raw_path)
    trajectories = loader.extract_marker_trajectories(data, fmt)
    return trajectories, raw_path

def load_processed(participant, condition):
    return writer.load_processed(participant, condition)

def dominant_freq(signal_1d, fs):
    nperseg = min(len(signal_1d), 4 * fs)
    f, pxx = welch(signal_1d, fs=fs, nperseg=nperseg)
    return float(f[np.argmax(pxx)])

print('✅ Setup done — pick a section below.')

---
## A — Raw inspection

Marker visibility report and Dos Z trajectories on a raw `.mat` file.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

trajectories, raw_path = load_raw(PARTICIPANT, CONDITION)
loader.print_visibility_report(raw_path, trajectories)

In [ ]:
# Visibility bar plot
summary = loader.summary_visibility(trajectories)
fig = plots.plot_marker_visibility(summary, title=f'{PARTICIPANT} / {CONDITION}')
plt.show()

In [ ]:
# Dos Z trajectories (raw)
fig = plots.plot_dos_trajectories(trajectories, PARTICIPANT, CONDITION)
plt.show()

---
## B — Crop window check

Verify that `crop_start_s` and `crop_end_s` (from `participants.py`) fall in sensible places on the raw signal.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

trajectories, _ = load_raw(PARTICIPANT, CONDITION)
trial = participants.get_trial(PARTICIPANT, CONDITION)

# Raw sacrum Z (no centering, no preprocessing)
dos_stack = [trajectories[m] for m in ('Dos01', 'Dos02', 'Dos03', 'Dos04') if m in trajectories]
sacrum_xyz = np.mean(np.stack(dos_stack, axis=0), axis=0)
z_raw = sacrum_xyz[:, 2]
t_raw = np.arange(len(z_raw)) / config.FS_RAW

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_raw, z_raw, color='steelblue', linewidth=0.6, alpha=0.8, label='raw sacrum Z')
if trial.crop_start_s is not None:
    ax.axvline(trial.crop_start_s, color='red', linestyle='--', linewidth=1.5,
               label=f'crop_start = {trial.crop_start_s}s')
if trial.crop_end_s is not None:
    ax.axvline(trial.crop_end_s, color='red', linestyle='--', linewidth=1.5,
               label=f'crop_end = {trial.crop_end_s}s')
if trial.crf_entry_s is not None:
    ax.axvspan(trial.crf_entry_s, trial.crf_exit_s, color='purple', alpha=0.2,
               label=f'CRF [{trial.crf_entry_s}, {trial.crf_exit_s}]s')
ax.set_xlabel('Time (s, raw reference)')
ax.set_ylabel('Z (mm)')
ax.set_title(f'{PARTICIPANT} / {CONDITION} — crop window check', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## C — Sanity check (processed)

Quick numerical health check on a processed `.npz`: cadence, dominant frequency, amplitude, NaN count.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

npz = load_processed(PARTICIPANT, CONDITION)
sig = npz['signal_final']
fs  = int(npz['fs_final'])

f_dom = dominant_freq(sig, fs)
amp_p5_p95 = np.percentile(sig, 95) - np.percentile(sig, 5)

report = pd.Series({
    'duration_s':     len(sig) / fs,
    'fs_final':       fs,
    'n_samples':      len(sig),
    'n_nan':          int(np.isnan(sig).sum()),
    'f_dom_hz':       round(f_dom, 3),
    'cadence_spm':    round(f_dom * 60, 1),
    'amp_p5_p95_mm':  round(amp_p5_p95, 2),
    'amp_range_mm':   round(np.ptp(sig), 2),
    'std_mm':         round(np.std(sig), 2),
    'emd_applied':    bool(npz['emd_applied']),
    'despike_applied': bool(npz['despike_applied']),
})

# Despike stats (stored as object array)
if 'despike_stats' in npz:
    stats = npz['despike_stats'].item() if npz['despike_stats'].dtype == object else None
    if stats is not None:
        report['n_spikes_interp'] = stats.get('n_spikes', np.nan)

print(f'{PARTICIPANT} / {CONDITION}')
report

---
## D — Despike diagnostic

Compare `sacrum_axis_raw` (before despike) and `sacrum_despiked` (after) to validate that no walking signal was lost.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

npz = load_processed(PARTICIPANT, CONDITION)
raw = npz['sacrum_axis_raw']
des = npz['sacrum_despiked']
t   = np.arange(len(raw)) / config.FS_RAW

stats = npz['despike_stats'].item() if npz['despike_stats'].dtype == object else {}
n_spikes = stats.get('n_spikes', 0)
threshold = stats.get('threshold', np.nan)
spike_idx = stats.get('indices_spike', np.array([]))

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(t, raw, color='steelblue', linewidth=0.5, label='raw')
if len(spike_idx) > 0:
    axes[0].scatter(t[spike_idx], raw[spike_idx], color='red', s=15, zorder=3,
                    label=f'{n_spikes} spike(s)')
axes[0].set_ylabel('Z (mm)')
axes[0].set_title(f'{PARTICIPANT} / {CONDITION}  —  raw  (threshold = ±{threshold:.1f} mm)' if not np.isnan(threshold)
                  else f'{PARTICIPANT} / {CONDITION}  —  raw')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t, des, color='forestgreen', linewidth=0.5, label='despiked')
axes[1].set_ylabel('Z (mm)')
axes[1].set_xlabel('Time (s, post-crop)')
axes[1].set_title('After despike (MAD-based linear interpolation)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Spikes interpolated: {n_spikes}  ({100*n_spikes/len(raw):.3f}%)')
print(f'Amplitude p5–p95 raw     : {np.percentile(raw,95)-np.percentile(raw,5):.2f} mm')
print(f'Amplitude p5–p95 despiked: {np.percentile(des,95)-np.percentile(des,5):.2f} mm')

---
## E — Preprocessing pipeline overview

Stacked plot of all signals in the chain: raw → despiked → EMD → filtered → final.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

npz = load_processed(PARTICIPANT, CONDITION)
t_raw   = np.arange(len(npz['sacrum_axis_raw'])) / config.FS_RAW
t_final = npz['time_final']

fig, axes = plt.subplots(5, 1, figsize=(14, 11), sharex=True)
panels = [
    ('sacrum_axis_raw', f"Raw barycentre Z  ({config.FS_RAW} Hz)",   'steelblue'),
    ('sacrum_despiked', "After despike (MAD)",                        'teal'),
    ('sacrum_emd',      "After EMD (adaptive walking-band selection)", 'darkorange'),
    ('sacrum_filt',     f"After Butterworth low-pass {config.BUTTER_CUTOFF_HZ} Hz", 'crimson'),
    ('signal_final',    f"Final signal ({int(npz['fs_final'])} Hz, decimated)", 'forestgreen'),
]

for ax, (key, label, color) in zip(axes, panels):
    sig = npz[key]
    t = t_final if key == 'signal_final' else t_raw
    ax.plot(t, sig, color=color, linewidth=0.5)
    ax.set_ylabel('mm', fontsize=9)
    ax.set_title(label, fontsize=10, fontweight='bold', loc='left')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time (s, post-crop)')
fig.suptitle(f'{PARTICIPANT} / {CONDITION} — preprocessing pipeline', fontweight='bold')
plt.tight_layout()
plt.show()

---
## F — 2D sacrum trajectory + sand bed

Sacrum XY plotted on top of the sand bed rectangle. Useful to visually confirm the bbox detection and inspect the loop geometry.

In [ ]:
PARTICIPANT = '012WaCh'
CONDITION   = 'silence'

npz = load_processed(PARTICIPANT, CONDITION)
dos_stack = [npz[f'raw_{m}'][:, :2] for m in ('Dos01', 'Dos02', 'Dos03', 'Dos04') if f'raw_{m}' in npz]
sacrum_xy = np.mean(np.stack(dos_stack, axis=0), axis=0)

det = detection.detect_sand_by_bbox(sacrum_xy)

corners = config.BAC_SAND_CORNERS_MM
rect = np.array([corners['A'], corners['B'], corners['C'], corners['D'], corners['A']])

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(sacrum_xy[:, 0], sacrum_xy[:, 1], '-', color='lightgray', linewidth=0.5,
        alpha=0.6, label='full trajectory')
ax.scatter(sacrum_xy[det['in_bbox'], 0], sacrum_xy[det['in_bbox'], 1],
           s=4, c='crimson', alpha=0.7, label=f"in-bbox  ({det['n_segments']} segment(s))")
ax.plot(rect[:, 0], rect[:, 1], 'k-', linewidth=2, label='sand bed')
for label, (x, y) in corners.items():
    ax.annotate(label, (x, y), fontsize=10, fontweight='bold',
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_title(f'{PARTICIPANT} / {CONDITION}  —  sacrum XY + sand bed', fontweight='bold')
ax.set_aspect('equal')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

if not np.isnan(det['entry_t']):
    print(f"Perturbation segment (post-crop): [{det['entry_t']:.1f}, {det['exit_t']:.1f}] s")
    print(f"Duration: {det['exit_t'] - det['entry_t']:.1f} s")
else:
    print('⚠️  No bbox segment detected')